# How to use regioinvent

Note that running this entire notebook will take you around 2 to 3 hours if you choose the highest cutoff option.

To be able to use regioinvent, you will need:
- to install the brightway2 Python library (brightway2 and NOT brightway2.5), easier is to get it through activity-browser: https://github.com/LCA-ActivityBrowser/activity-browser
- a brightway project within which there is an ecoinvent database with either the version 3.9/3.9.1/3.10/3.10.1 cutoff
- to download the latest version of the trade database stored here: https://doi.org/10.5281/zenodo.11583814

In [1]:
import pandas as pd
import numpy as np
import sys
# change the path here to wherever you stored the Regioinvent Python package 
# only needed if you've not installed through pip
#sys.path.append('C://Users/11max/PycharmProjects/Regioinvent/src/')
import regioinvent

13:52:49+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
import bw2data
bw2data.projects.set_current("ecoinvent-3.10-cutoff")

In [3]:
bw2data.databases

Databases dictionary with 36 objects, including:
	Aluminium
	Antimony
	Arsenic
	Beryllium
	Bismuth
	Boron
	Cadmium
	Chromium
	Cobalt
	Copper
Use `list(this object)` to get the complete list.

In [3]:
#bw2io.ecoinvent.import_ecoinvent_release(
#    version="3.10.1",
#    system_model="cutoff",
#    username="xxx",
#    password="xxx",
#    biosphere_name="biosphere3",
#)

The initialization of the Regioinvent class requires 3 arguments:
- the name of the brightway2 project where you stored ecoinvent and where regioinvent will be added
- the name you gave your ecoinvent database within your brightway2 project
- the version of the ecoinvent database ('3.9', '3.9.1', '3.10' or '3.10.1')

In [4]:
regio = regioinvent.Regioinvent(
    bw_project_name="ecoinvent-3.10-cutoff",
    ecoinvent_database_name='ecoinvent-3.10.1-cutoff',
    ecoinvent_version='3.10.1'
)

First step of regioinvent is to spatialize the original ecoinvent database. This entails two steps:
- Creating a new biosphere database for spatialized elementary flows (e.g., Ammonia, CA-QC)
- Spatializing the elementary flows used within the ecoinvent database, based on the location of the process itself

In [5]:
regio.spatialize_my_ecoinvent()

2026-04-23 13:53:16,829 - Regioinvent - INFO - Creating spatialized biosphere flows...
100%|████████████████████████████████| 110559/110559 [00:02<00:00, 42824.03it/s]

13:53:20+0200 [info     ] Vacuuming database            



2026-04-23 13:53:35,092 - Regioinvent - INFO - Extracting ecoinvent to wurst...


Getting activity data


100%|█████████████████████████████████| 23523/23523 [00:00<00:00, 253557.09it/s]


Adding exchange data to activities


100%|████████████████████████████████| 743409/743409 [00:33<00:00, 22036.05it/s]


Filling out exchange data


100%|███████████████████████████████████| 23523/23523 [00:02<00:00, 8089.88it/s]
2026-04-23 13:54:19,034 - Regioinvent - INFO - Spatializing ecoinvent...


At this stage, regioinvent writes only one database to your brightway2 project:
- "_biosphere3_spatialized_flows_", which contains all the newly created spatialized elementary flows

The spatialized copy of ecoinvent is prepared fully in memory at this step. It is written later, together with the regioinvent database, once both databases have been fully relinked.

Because elementary flows are now spatialized, you will need a specific LCIA method to operate with any flow (spatialized or not). The following function imports such methods. Currently available: "IW v2.1", "EF v3.1", "ReCiPe 2016 v1.03 (H)". Can also import all of them in one go.

In [6]:
regio.import_fully_regionalized_impact_method()

2026-04-23 13:54:32,673 - Regioinvent - INFO - Importing all available fully regionalized lcia methods for ecoinvent3.10.


UnknownObject: 

If you want to go further in the regionalization, i.e., to create new national production processes and to rely on trade data to create regionalized consumption markets of ecoinvent, you can run the next function. There are 3 arguments:
- _trade_database_path_ which is the path where you stored the trade database you downloaded from Zenodo: https://doi.org/10.5281/zenodo.11583814
- _target_database_name_ which is the name that the created regioinvent database will take
- _cutoff_ which is the cut-off (between 0 and 1) beyond which countries will be aggreated as RoW. For more info on what cutoff does, see section "Selection of countries for regionalization" of the Methodology.md file.

This function now runs the in-memory relinking and the final Brightway write for both dependent databases. You no longer need to call `write_database()` separately afterwards.

In [ ]:
regio.regionalize_ecoinvent_with_trade(
    trade_database_path='/Users/romain/GitHub/Regioinvent/dev/trade_data.db',
    target_database_name='Regioinvent',
    cutoff=0.99
)

2026-04-23 13:55:45,528 - Regioinvent - INFO - Extracting and formatting trade data...
2026-04-23 13:55:56,166 - Regioinvent - INFO - Regionalizing main inputs of internationally-traded products of ecoinvent...
100%|███████████████████████████████████████| 1982/1982 [00:51<00:00, 38.84it/s]
2026-04-23 13:56:47,245 - Regioinvent - INFO - Regionalizing main inputs of non-internationally traded processes of ecoinvent...
100%|███████████████████████████████████████████| 67/67 [00:16<00:00,  3.96it/s]
2026-04-23 13:57:04,253 - Regioinvent - INFO - Creating consumption markets for internationally-traded products...
100%|███████████████████████████████████████| 1982/1982 [00:55<00:00, 35.61it/s]
2026-04-23 13:58:02,595 - Regioinvent - INFO - Link regioinvent processes to each other...
2026-04-23 13:58:24,923 - Regioinvent - INFO - Aggregate duplicates together...
2026-04-23 13:58:34,647 - Regioinvent - INFO - Regionalizing the elementary flows of the regioinvent database...
2026-04-23 13:58:3

13:58:42+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|████████████████████████████████████| 23523/23523 [01:06<00:00, 355.20it/s]


13:59:49+0200 [info     ] Vacuuming database            


2026-04-23 14:00:06,663 - Regioinvent - INFO - Writing Brightway database 'Regioinvent' without processing...


14:00:09+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


  3%|█▏                                  | 7168/228070 [00:33<14:35, 252.37it/s]

This call writes both final databases to your brightway project: the spatialized ecoinvent copy (`the-name-of-your-ecoinvent-database - regionalized`) and the regioinvent database (`Regioinvent` in the example above). You can then go on brightway2 or AB to perform your LCAs normally with regioinvent.

In [10]:
len(bw2data.Database("ecoinvent-3.10.1-cutoff - regionalized"))

251593

In [11]:
len(bw2data.Database("Regioinvent"))

0

In [3]:
import bw2calc

In [4]:
db = bw2data.Database("Regioinvent")
results = []
for _ in range(6):
    print("yes")
    ds = db.random()
    lca = bw2calc.LCA({ds:1}, method=('IPCC 2021', 'climate change: fossil, including SLCFs', 'global warming potential (GWP100)'))
    lca.lci()
    lca.lcia()
    results.append(
        [
            ds["name"],
            ds["reference product"],
            ds["location"],
            lca.score
        ]
    )

import pandas as pd

pd.DataFrame(results)

yes


/opt/homebrew/Caskroom/miniforge/base/envs/regioinvent/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 7.59e+13)
  warnings.warn(msg, UmfpackWarning)


yes
yes
yes
yes
yes


,0,1,2,3
0,"consumption market for ventilation system, dec...","ventilation system, decentralized, 6 x 120 m3/...",IN,11111.221433
1,consumption market for gold,gold,ZW,49751.993915
2,consumption market for ventilation of dwelling...,"ventilation of dwellings, central, 1 x 720 m3/h",TZ,3.029299
3,consumption market for photovoltaic slanted-ro...,"photovoltaic slanted-roof installation, 3kWp, ...",FR,8066.131281
4,consumption market for isoproturon,isoproturon,IE,6.717141
5,consumption market for anthranilic acid,anthranilic acid,CZ,5.532118


In [5]:
results

[['consumption market for ventilation system, decentralized, 6 x 120 m3/h, polyethylene ducts, with earth tube heat exchanger',
  'ventilation system, decentralized, 6 x 120 m3/h, polyethylene ducts, with earth tube heat exchanger',
  'IN',
  11111.221433415523],
 ['consumption market for gold', 'gold', 'ZW', 49751.99391503114],
 ['consumption market for ventilation of dwellings, central, 1 x 720 m3/h',
  'ventilation of dwellings, central, 1 x 720 m3/h',
  'TZ',
  3.0292988252818187],
 ['consumption market for photovoltaic slanted-roof installation, 3kWp, single-Si, laminated, integrated, on roof',
  'photovoltaic slanted-roof installation, 3kWp, single-Si, laminated, integrated, on roof',
  'FR',
  8066.13128068786],
 ['consumption market for isoproturon',
  'isoproturon',
  'IE',
  6.717140836099581],
 ['consumption market for anthranilic acid',
  'anthranilic acid',
  'CZ',
  5.532117529197189]]